Input: 1 × 32 × 32
       ↓
       
C1: Conv2d(1, 6, kernel_size=5)
       → 6 × 28 × 28
       ↓
S2: AvgPool2d(2, stride=2)
       → 6 × 14 × 14
       ↓
C3: Conv2d(6, 16, kernel_size=5)
       → 16 × 10 × 10
       ↓
S4: AvgPool2d(2, stride=2)
       → 16 × 5 × 5
       ↓
C5: Conv2d(16, 120, kernel_size=5)
       → 120 × 1 × 1
       ↓
Flatten
       → 120
       ↓
F6: Linear(120, 84)
       → 84
       ↓
Output: Linear(84, 10)
       → 10 classes

In [3]:

import torch
import torch.nn as nn


class LeNet5(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            # C1: 1x32x32 -> 6x28x28
            nn.Conv2d(1, 6, kernel_size=5),
            nn.Tanh(),

            # S2: 6x28x28 -> 6x14x14
            nn.AvgPool2d(kernel_size=2, stride=2),

            # C3: 6x14x14 -> 16x10x10
            nn.Conv2d(6, 16, kernel_size=5),
            nn.Tanh(),

            # S4: 16x10x10 -> 16x5x5
            nn.AvgPool2d(kernel_size=2, stride=2),

            # C5: 16x5x5 -> 120x1x1
            nn.Conv2d(16, 120, kernel_size=5),
            nn.Tanh()
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(120, 84),
            nn.Tanh(),
            nn.Linear(84, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = LeNet5().to(device)

x = torch.randn(1, 1, 32, 32).to(device)

from torchinfo import summary

model = LeNet5().to(device)

summary(
    model,
    input_size=(1, 1, 32, 32),
    device=device,
    col_names=[
        "input_size",
        "output_size",
        "num_params",
        "kernel_size",
        "mult_adds"
    ],
    depth=3
)

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #                   Kernel Shape              Mult-Adds
LeNet5                                   [1, 1, 32, 32]            [1, 10]                   --                        --                        --
├─Sequential: 1-1                        [1, 1, 32, 32]            [1, 120, 1, 1]            --                        --                        --
│    └─Conv2d: 2-1                       [1, 1, 32, 32]            [1, 6, 28, 28]            156                       [5, 5]                    122,304
│    └─Tanh: 2-2                         [1, 6, 28, 28]            [1, 6, 28, 28]            --                        --                        --
│    └─AvgPool2d: 2-3                    [1, 6, 28, 28]            [1, 6, 14, 14]            --                        2                         --
│    └─Conv2d: 2-4                       [1, 6, 14, 14]            [1, 16, 10, 10]           2,416  